In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.special import erf, erfc
from scipy.optimize import brentq
from scipy.integrate import solve_ivp
from IPython.display import HTML

# --- Параметры материалов (лёд / вода) ---
rho = 950.0          # плотность, кг/м³
lam = 333000.0       # скрытая теплота плавления, Дж/кг
T0  = -10.0          # температура на левой границе (лёд), °C
T2  = 5.0            # начальная температура воды, °C

k1, c1 = 2.22, 2090.0   # теплопроводность и теплоёмкость льда
k2, c2 = 0.56, 4180.0   # то же для воды
a1 = np.sqrt(k1 / (rho * c1))   # температуропроводность льда
a2 = np.sqrt(k2 / (rho * c2))   # температуропроводность воды

# --- Сетка ---
T_end = 72000.0      # 20 часов
L     = 0.40         # длина области, м (с запасом)
dx    = 0.004        # шаг сетки, м

Nx = int(L / dx) + 1
x  = np.linspace(0, L, Nx)
print(f'Nx = {Nx}')

Nx = 101


In [2]:

def stefan_eq(alpha):
    z1 = alpha / (2 * a1)
    z2 = alpha / (2 * a2)
    t1 = k1 * abs(T0) / (a1 * np.sqrt(np.pi) * erf(z1)) * np.exp(-z1**2)   # поток из льда
    t2 = k2 * T2 / (a2 * np.sqrt(np.pi) * erfc(z2)) * np.exp(-z2**2)       # поток из воды
    return t1 - t2 - lam * rho * alpha / 2    # = 0 при верном alpha

# Z(t) = alpha * sqrt(t)
alpha = brentq(stefan_eq, 1e-6, 0.002)

#  U = A + B * erf(x / 2a√t)
A1 = T0
B1 = -T0 / erf(alpha / (2 * a1))
B2 = T2 / erfc(alpha / (2 * a2))
A2 = T2 - B2

print(f'α  = {alpha:.6e} м/√s')
print(f'Z(T_end) = {alpha * np.sqrt(T_end) * 100:.2f} см')
print(f'A1={A1:.2f}  B1={B1:.2f}  A2={A2:.4f}  B2={B2:.4f}')


# лёд (x<=Z) и вода (x>Z) — разные erf-профили
def solve_analytical(x_arr, t_val):
    if t_val <= 0:
        res = np.full_like(x_arr, T2)
        res[0] = T0
        return res
    Z = alpha * np.sqrt(t_val)        # положение фронта
    res = np.empty_like(x_arr)
    ice = x_arr <= Z
    res[ice]  = A1 + B1 * erf(x_arr[ice]  / (2 * a1 * np.sqrt(t_val)))
    res[~ice] = A2 + B2 * erf(x_arr[~ice] / (2 * a2 * np.sqrt(t_val)))
    res[0] = T0
    return res

α  = 3.507938e-04 м/√s
Z(T_end) = 9.41 см
A1=-10.00  B1=53.92  A2=-4.8249  B2=9.8249


In [3]:
# T(H)
def T_from_H(H):
    T     = np.empty_like(H)
    ice   = H < 0
    mushy = (H >= 0) & (H <= rho * lam)
    water = H > rho * lam
    T[ice]   = H[ice] / (rho * c1)
    T[mushy] = 0.0
    T[water] = (H[water] - rho * lam) / (rho * c2)
    return T


# Правая часть ОДУ
def enthalpy_rhs(H):
    T = T_from_H(H)
    T[0]  = T0
    T[-1] = T2
    k_cur  = np.where(T <= 0, k1, k2)
    k_half = (k_cur[:-1] + k_cur[1:]) / 2
    flux   = k_half * np.diff(T) / dx
    dHdt = np.zeros_like(H)
    dHdt[1:-1] = np.diff(flux) / dx
    return dHdt


def rk4_step(H, dt):
    k1 = enthalpy_rhs(H)
    k2 = enthalpy_rhs(H + 0.5 * dt * k1)
    k3 = enthalpy_rhs(H + 0.5 * dt * k2)
    k4 = enthalpy_rhs(H + dt * k3)
    return H + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)


def solve_numerical(x, frame_times):
    Nx = len(x)
    dt_num = 5.0   # шаг по времени 

    H = np.full(Nx, rho * lam + rho * c2 * T2)   # начало: вода при T2
    H[0] = rho * c1 * T0                           # левый край: лёд при T0

    frames = {}
    fi = 0
    if frame_times[0] == 0:
        frames[0] = T_from_H(H.copy()); frames[0][0] = T0
        fi = 1

    t_cur = 0.0
    while fi < len(frame_times):
        t_cur += dt_num
        H = rk4_step(H, dt_num)
        if t_cur >= frame_times[fi] - dt_num / 2:
            T = T_from_H(H.copy())
            T[0] = T0
            frames[fi] = T
            fi += 1

    return frames

In [4]:
dt_frame = 600.0     
frame_times = np.arange(0, T_end + dt_frame / 2, dt_frame)
n_frames = len(frame_times)

x_lim = 0.25         
x_plot = x

T_num = solve_numerical(x, frame_times)

In [5]:
plt.rcParams['animation.embed_limit'] = 30

fig, ax = plt.subplots(figsize=(10, 5))
ln_num, = ax.plot([], [], 'b-',  lw=2,   label='Сетка')
ln_an,  = ax.plot([], [], 'g-.', lw=2,   label='Полуаналитика')
txt = ax.text(0.02, 0.05, '', transform=ax.transAxes, fontsize=11,
              verticalalignment='bottom',
              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlim(0, x_lim)
ax.set_ylim(T0 - 2, T2 + 2)
ax.axhline(0, color='k', lw=0.5, alpha=0.3)
ax.set_xlabel('x')
ax.set_ylabel('T')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')


def animate(frame):
    tv = frame_times[frame]
    ln_num.set_data(x_plot, T_num[frame])
    ln_an.set_data(x_plot, solve_analytical(x_plot, tv))
    hrs = tv / 3600
    txt.set_text(f't = {hrs:.1f} ч')
    return ln_num, ln_an, txt


anim = FuncAnimation(fig, animate, frames=n_frames,
                     interval=100, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())